# HW14: Retrieval, FAISS, mini-RAG

Домашнее задание по теме: эмбеддинги, индекс FAISS, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

**База знаний:** главы из "Капитанской дочки" А.С. Пушкина и "Отцы и дети" И.С. Тургенева.

---

## 1. Импорты, seed и среда

Импортируем необходимые библиотеки, фиксируем seed, определяем устройство для torch (если используется).

In [25]:
# Импорты и настройки среды
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Optional

# Для FAISS и эмбеддингов
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS не установлен. Установите faiss-cpu для работы индекса.")

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if 'torch' in sys.modules:
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)

print(f"Используемое устройство: {DEVICE}")
print(f"FAISS доступен: {FAISS_AVAILABLE}")

Используемое устройство: cpu
FAISS доступен: True


## 2. База знаний и первичный анализ

В качестве базы знаний используются главы из "Капитанской дочки" А.С. Пушкина и "Отцы и дети" И.С. Тургенева. Каждая глава — отдельный документ.

In [26]:
# Пути к текстам
pushkin_path = r"Pushkin.txt"
turgenev_path = r"Turgenev.txt"

# Функции для разбиения на главы
def split_pushkin(text):
    import re
    pattern = r'(ГЛАВА\s+[IVXLCDM]+\.)'
    chapters = re.split(pattern, text)
    result = []
    for i in range(1, len(chapters), 2):
        result.append(chapters[i+1].strip())
    return result

def split_turgenev(text):
    import re
    pattern = r'(Глава\s+\d+)'
    chapters = re.split(pattern, text)
    result = []
    for i in range(1, len(chapters), 2):
        result.append(chapters[i+1].strip())
    return result

# Загрузка текстов
def load_documents():
    with open(pushkin_path, encoding='utf-8') as f:
        pushkin_text = f.read()
    with open(turgenev_path, encoding='utf-8') as f:
        turgenev_text = f.read()
    pushkin_docs = split_pushkin(pushkin_text)
    turgenev_docs = split_turgenev(turgenev_text)
    docs = []
    for i, ch in enumerate(pushkin_docs):
        docs.append({'doc_id': f'pushkin_{i+1}', 'title': f'Капитанская дочка, глава {i+1}', 'text': ch})
    for i, ch in enumerate(turgenev_docs):
        docs.append({'doc_id': f'turgenev_{i+1}', 'title': f'Отцы и дети, глава {i+1}', 'text': ch})
    return docs

documents = load_documents()
documents_df = pd.DataFrame(documents)

print(f"Всего документов: {len(documents_df)}")
display(documents_df.head())

# Покажем 3 примера
for i in range(3):
    print(f"--- {documents[i]['title']} ---\n{documents[i]['text'][:400]}\n...")

Всего документов: 42


,doc_id,title,text
0,pushkin_1,"Капитанская дочка, глава 1",СЕРЖАНТ ГВАРДИИ.\n\n\n\n - Был бы гвардии ...
1,pushkin_2,"Капитанская дочка, глава 2","ВОЖАТЫЙ\n\n\n\n Сторона ль моя, сторонушка..."
2,pushkin_3,"Капитанская дочка, глава 3","КРЕПОСТЬ.\n\n\n\n Мы в фортеции живем,\n ..."
3,pushkin_4,"Капитанская дочка, глава 4","ПОЕДИНОК.\n\n\n\n - Ин изволь, и стань же ..."
4,pushkin_5,"Капитанская дочка, глава 5","ЛЮБОВЬ.\n\n\n\n Ах ты, девка, девка красна..."


--- Капитанская дочка, глава 1 ---
СЕРЖАНТ ГВАРДИИ.



     - Был бы гвардии он завтра ж капитан.
     - Того не надобно; пусть в армии послужит.
     - Изрядно сказано! пускай его потужит...
     .....................................
     Да кто его отец?
     Княжнин.


     Отец мой  Андрей  Петрович  Гринев в  молодости своей служил  при графе
Минихе,  и вышел в отставку  премьер-маиором в 17.. году. С тех пор жил он в
своей  С
...
--- Капитанская дочка, глава 2 ---
ВОЖАТЫЙ



     Сторона ль моя, сторонушка,
     Сторона незнакомая!
     Что не сам ли я на тебя зашел,
     Что не добрый ли да меня конь завез:
     Завезла меня, доброго молодца,
     Прытость, бодрость молодецкая,
     И хмелинушка кабацкая.
     Старинная песня
     Дорожные  размышления мои  были не  очень  приятны.  Проигрыш  мой,  по
тогдашним ценам,  был  немаловажен.  Я  не  мог  не  пр
...
--- Капитанская дочка, глава 3 ---
КРЕПОСТЬ.



     Мы в фортеции живем,
     Хлеб едим и воду пьем;
     А как лютые в

## 3. Чанкинг документов

Реализуем разбиение документов на текстовые фрагменты (чанки) с overlap. Покажем примеры чанков и объясним выбранные параметры.

In [27]:
# Чанкинг: разбиваем текст на фрагменты по N слов с overlap

def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

# Применим чанкинг к одному документу и покажем примеры
chunk_size = 30
overlap = 10
example_doc = documents[0]['text']
chunks = chunk_text(example_doc, chunk_size=chunk_size, overlap=overlap)
print(f"Документ разбит на {len(chunks)} чанков (chunk_size={chunk_size}, overlap={overlap})\n")
for i, ch in enumerate(chunks[:3]):
    print(f"Чанк {i+1}:\n{ch}\n---")

Документ разбит на 116 чанков (chunk_size=30, overlap=10)

Чанк 1:
СЕРЖАНТ ГВАРДИИ. - Был бы гвардии он завтра ж капитан. - Того не надобно; пусть в армии послужит. - Изрядно сказано! пускай его потужит... ..................................... Да кто его отец? Княжнин.
---
Чанк 2:
сказано! пускай его потужит... ..................................... Да кто его отец? Княжнин. Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе, и вышел в отставку премьер-маиором в 17.. году.
---
Чанк 3:
графе Минихе, и вышел в отставку премьер-маиором в 17.. году. С тех пор жил он в своей Симбирской деревни, где и женился на девице Авдотьи Васильевне Ю., дочери бедного тамошнего
---


**Пояснение:**
- `chunk_size = 40` — размер чанка (слов)
- `overlap = 10` — перекрытие между чанками (слов)

Такие параметры позволяют сохранить смысловые связи между чанками и не терять важную информацию на границах.

## 4. Эмбеддинги и индекс FAISS

Построим векторные представления чанков и FAISS-индекс. Покажем примеры поиска top-k фрагментов по запросу.

In [28]:
# Чанкинг всей базы знаний
all_chunks = []
for doc in documents:
    doc_chunks = chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap)
    for idx, chunk in enumerate(doc_chunks):
        all_chunks.append({
            'doc_id': doc['doc_id'],
            'title': doc['title'],
            'chunk_id': f"{doc['doc_id']}_chunk_{idx}",
            'chunk_text': chunk
        })
chunks_df = pd.DataFrame(all_chunks)
print(f"Всего чанков: {len(chunks_df)}")
display(chunks_df.head())

# Эмбеддинги: SentenceTransformers или TF-IDF fallback
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=device)
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, batch_size=16, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, batch_size=16, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True).astype('float32')

class TfidfBackend(EmbeddingBackend):
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype('float32')
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype('float32')
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

def select_backend(device: str = "cpu") -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
        print("Используется SentenceTransformers.")
        return backend
    except Exception as e:
        print("SentenceTransformers недоступен, fallback на TF-IDF.")
        print("Причина:", repr(e))
        return TfidfBackend()

backend = select_backend(DEVICE)
chunk_vectors = backend.fit_documents(chunks_df['chunk_text'].tolist())

# FAISS-индекс
if not FAISS_AVAILABLE:
    raise RuntimeError("FAISS не установлен. Установите faiss-cpu.")
index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)

# Поиск top-k чанков по запросу
def search_chunks(query: str, top_k: int = 3):
    query_vec = backend.encode_queries([query]).astype('float32')
    scores, indices = index.search(query_vec, top_k)
    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append({
            'rank': rank,
            'score': float(score),
            'doc_id': chunk_row['doc_id'],
            'title': chunk_row['title'],
            'chunk_text': chunk_row['chunk_text']
        })
    return pd.DataFrame(rows)

# Примеры поиска
test_queries = [
    "Кто такой Савельич?",
    "Главная тема романа Отцы и дети",
    "Что случилось с Базаровым?"
]
for q in test_queries:
    print(f"\nЗапрос: {q}")
    display(search_chunks(q, top_k=3))

Всего чанков: 4413


,doc_id,title,chunk_id,chunk_text
0,pushkin_1,"Капитанская дочка, глава 1",pushkin_1_chunk_0,СЕРЖАНТ ГВАРДИИ. - Был бы гвардии он завтра ж ...
1,pushkin_1,"Капитанская дочка, глава 1",pushkin_1_chunk_1,сказано! пускай его потужит... ..................
2,pushkin_1,"Капитанская дочка, глава 1",pushkin_1_chunk_2,"графе Минихе, и вышел в отставку премьер-маиор..."
3,pushkin_1,"Капитанская дочка, глава 1",pushkin_1_chunk_3,"и женился на девице Авдотьи Васильевне Ю., доч..."
4,pushkin_1,"Капитанская дочка, глава 1",pushkin_1_chunk_4,сестры умерли во младенчестве. Матушка была ещ...


Используется SentenceTransformers.


Batches: 100%|██████████| 276/276 [00:31<00:00,  8.73it/s]


Запрос: Кто такой Савельич?


,rank,score,doc_id,title,chunk_text
0,1,0.673854,pushkin_1,"Капитанская дочка, глава 1","сто рублей. ""Как! зачем?"" - спросил изумленный..."
1,2,0.658123,pushkin_7,"Капитанская дочка, глава 7",остановились. Гляжу: Савельич лежит в ногах у ...
2,3,0.654460,pushkin_13,"Капитанская дочка, глава 13","Савельич от меня не отставал, поговаривая про ..."



Запрос: Главная тема романа Отцы и дети


,rank,score,doc_id,title,chunk_text
0,1,0.558695,pushkin_10,"Капитанская дочка, глава 10","оная состояла, читатель увидит из следующей гл..."
1,2,0.526647,turgenev_4,"Отцы и дети, глава 4",человек. -- Архаическое явление! А отец у тебя...
2,3,0.512119,turgenev_1,"Отцы и дети, глава 1","дяди с материнской стороны, Ильи Колязина, важ..."



Запрос: Что случилось с Базаровым?


,rank,score,doc_id,title,chunk_text
0,1,0.761331,turgenev_28,"Отцы и дети, глава 28","великим, толчется в Петербурге и, по его увере..."
1,2,0.740774,pushkin_8,"Капитанская дочка, глава 8",Нечего сказать! бедный Иван Кузмич! кто бы под...
2,3,0.730622,turgenev_27,"Отцы и дети, глава 27","худо. А к той, помнишь? послал? -- Послал, как..."


## 5. Контрольные запросы и оценка retrieval

Составим набор контрольных запросов, укажем ожидаемые источники, выполним retrieval и посчитаем метрики hit@k, recall@k. Результаты сохранятся в artifacts/retrieval_eval.csv.

In [29]:
# Контрольные запросы и ожидаемые источники
benchmark_queries = [
    {"query": "Кто такой Савельич?", "expected_source": "pushkin_1"},
    {"query": "Что случилось с Базаровым?", "expected_source": "turgenev_28"},
    {"query": "Главная тема романа Отцы и дети", "expected_source": "turgenev_1"},
    {"query": "Как Петр Гринев попал в армию?", "expected_source": "pushkin_1"},
    {"query": "Кто такая Марья Ивановна?", "expected_source": "pushkin_5"},
    {"query": "Как Аркадий познакомился с Базаровым?", "expected_source": "turgenev_2"},
    {"query": "Что произошло на дуэли?", "expected_source": "turgenev_18"},
    {"query": "Как погиб Пугачев?", "expected_source": "pushkin_12"},
    {"query": "Кто был отцом Аркадия?", "expected_source": "turgenev_1"},
    {"query": "Как закончилась история Гринева?", "expected_source": "pushkin_14"}
]

def unique_doc_order(result_df):
    seen = set()
    ordered = []
    for doc_id in result_df['doc_id'].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered

results = []
top_k = 3
for row in benchmark_queries:
    result_df = search_chunks(row['query'], top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)
    hit = int(row['expected_source'] in predicted_doc_ids)
    recall = int(row['expected_source'] in predicted_doc_ids)  # по одному релевантному
    rank = predicted_doc_ids.index(row['expected_source']) + 1 if row['expected_source'] in predicted_doc_ids else None
    results.append({
        'query': row['query'],
        'expected_source': row['expected_source'],
        'retrieved_sources': ', '.join(predicted_doc_ids),
        'hit_at_k': hit,
        'rank_of_first_relevant': rank
    })
retrieval_eval_df = pd.DataFrame(results)
display(retrieval_eval_df)

# Сохраняем в артефакт
retrieval_eval_df.to_csv('artifacts/retrieval_eval.csv', index=False, encoding='utf-8')

# Метрики
mean_hit = retrieval_eval_df['hit_at_k'].mean()
print(f"Средний hit@{top_k}: {mean_hit:.2f}")

,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,Кто такой Савельич?,pushkin_1,"pushkin_1, pushkin_7, pushkin_13",1,1.0
1,Что случилось с Базаровым?,turgenev_28,"turgenev_28, pushkin_8, turgenev_27",1,1.0
2,Главная тема романа Отцы и дети,turgenev_1,"pushkin_10, turgenev_4, turgenev_1",1,3.0
3,Как Петр Гринев попал в армию?,pushkin_1,"turgenev_24, pushkin_14, turgenev_1",0,NaN
4,Кто такая Марья Ивановна?,pushkin_5,"pushkin_12, pushkin_8, pushkin_14",0,NaN
5,Как Аркадий познакомился с Базаровым?,turgenev_2,"turgenev_9, turgenev_14, turgenev_21",0,NaN
6,Что произошло на дуэли?,turgenev_18,"turgenev_22, pushkin_7, turgenev_21",0,NaN
7,Как погиб Пугачев?,pushkin_12,"pushkin_14, pushkin_12, pushkin_11",1,2.0
8,Кто был отцом Аркадия?,turgenev_1,"turgenev_4, turgenev_14, turgenev_7",0,NaN
9,Как закончилась история Гринева?,pushkin_14,"pushkin_14, turgenev_24",1,1.0


Средний hit@3: 0.50


## 6. Эксперимент с параметрами retrieval

Сравним качество retrieval при двух разных значениях chunk_size (например, 40 и 80).

In [ ]:
# Сравнение двух chunk_size
experiment_results = []
for exp_chunk_size in [40, 80]:
    # Чанкинг
    exp_chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc['text'], chunk_size=exp_chunk_size, overlap=overlap)
        for idx, chunk in enumerate(doc_chunks):
            exp_chunks.append({
                'doc_id': doc['doc_id'],
                'title': doc['title'],
                'chunk_id': f"{doc['doc_id']}_chunk_{idx}",
                'chunk_text': chunk
            })
    exp_chunks_df = pd.DataFrame(exp_chunks)
    # Эмбеддинги
    exp_vectors = backend.fit_documents(exp_chunks_df['chunk_text'].tolist())
    # FAISS
    exp_index = faiss.IndexFlatIP(exp_vectors.shape[1])
    exp_index.add(exp_vectors)
    # Retrieval
    def exp_search_chunks(query, top_k=3):
        query_vec = backend.encode_queries([query]).astype('float32')
        scores, indices = exp_index.search(query_vec, top_k)
        rows = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            chunk_row = exp_chunks_df.iloc[int(idx)]
            rows.append(chunk_row['doc_id'])
        return rows
    hits = []
    for row in benchmark_queries:
        pred = exp_search_chunks(row['query'], top_k=3)
        hit = int(row['expected_source'] in pred)
        hits.append(hit)
    mean_hit = np.mean(hits)
    experiment_results.append({'chunk_size': exp_chunk_size, 'mean_hit@3': mean_hit, 'num_chunks': len(exp_chunks_df)})

exp_df = pd.DataFrame(experiment_results)
display(exp_df)

print("\nВывод: увеличение chunk_size уменьшает количество чанков, но может снизить качество, если смысловые фрагменты становятся слишком большими.")

Batches: 100%|██████████| 80/80 [00:16<00:00,  4.74it/s]


,chunk_size,mean_hit@3,num_chunks
0,40,0.4,2947
1,80,0.3,1276



Вывод: увеличение chunk_size уменьшает количество чанков, но может снизить качество, если смысловые фрагменты становятся слишком большими.


## 7. Обновление базы знаний и переиндексация

Добавим новые документы, переиндексируем базу и сравним результаты retrieval до и после обновления. Сохраним сравнение в artifacts/retrieval_before_after_update.csv.

In [31]:
# Добавим новые документы
new_docs = [
    {'doc_id': 'pushkin_new1', 'title': 'Капитанская дочка, эпилог', 'text': 'Повествование о дальнейшей судьбе героев и итогах событий.'},
    {'doc_id': 'turgenev_new1', 'title': 'Отцы и дети, эпилог', 'text': 'Описание жизни Аркадия и его семьи после смерти Базарова.'},
    {'doc_id': 'pushkin_new2', 'title': 'Капитанская дочка, анализ', 'text': 'Краткий анализ основных тем и мотивов романа.'}
]
updated_documents = documents + new_docs

# Чанкинг и индексация обновлённой базы
def build_chunks_and_index(docs, chunk_size=30, overlap=10):
    all_chunks = []
    for doc in docs:
        doc_chunks = chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap)
        for idx, chunk in enumerate(doc_chunks):
            all_chunks.append({
                'doc_id': doc['doc_id'],
                'title': doc['title'],
                'chunk_id': f"{doc['doc_id']}_chunk_{idx}",
                'chunk_text': chunk
            })
    df = pd.DataFrame(all_chunks)
    vectors = backend.fit_documents(df['chunk_text'].tolist())
    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors)
    return df, idx

# Индекс до обновления
chunks_df_before, index_before = build_chunks_and_index(documents, chunk_size=chunk_size, overlap=overlap)
# Индекс после обновления
chunks_df_after, index_after = build_chunks_and_index(updated_documents, chunk_size=chunk_size, overlap=overlap)

def search_chunks_df(query, chunks_df, index, top_k=3):
    query_vec = backend.encode_queries([query]).astype('float32')
    scores, indices = index.search(query_vec, top_k)
    doc_ids = []
    for idx in indices[0]:
        doc_ids.append(chunks_df.iloc[int(idx)]['doc_id'])
    return doc_ids

# Сравнение retrieval до и после обновления
compare_results = []
for row in benchmark_queries:
    before = search_chunks_df(row['query'], chunks_df_before, index_before, top_k=3)
    after = search_chunks_df(row['query'], chunks_df_after, index_after, top_k=3)
    changed = before != after
    compare_results.append({
        'query': row['query'],
        'before_retrieved_sources': ', '.join(before),
        'after_retrieved_sources': ', '.join(after),
        'changed': changed
    })
compare_df = pd.DataFrame(compare_results)
display(compare_df)

# Сохраняем в артефакт
compare_df.to_csv('artifacts/retrieval_before_after_update.csv', index=False, encoding='utf-8')

Batches: 100%|██████████| 276/276 [00:31<00:00,  8.88it/s]


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Кто такой Савельич?,"pushkin_1, pushkin_7, pushkin_13","pushkin_1, pushkin_7, pushkin_13",False
1,Что случилось с Базаровым?,"turgenev_28, pushkin_8, turgenev_27","turgenev_28, pushkin_8, turgenev_27",False
2,Главная тема романа Отцы и дети,"pushkin_10, turgenev_4, turgenev_1","pushkin_new2, pushkin_10, turgenev_4",True
3,Как Петр Гринев попал в армию?,"turgenev_24, pushkin_14, turgenev_1","turgenev_24, pushkin_14, turgenev_1",False
4,Кто такая Марья Ивановна?,"pushkin_12, pushkin_8, pushkin_14","pushkin_12, pushkin_8, pushkin_14",False
5,Как Аркадий познакомился с Базаровым?,"turgenev_9, turgenev_14, turgenev_21","turgenev_9, turgenev_14, turgenev_21",False
6,Что произошло на дуэли?,"turgenev_22, pushkin_7, turgenev_21","turgenev_22, pushkin_7, turgenev_21",False
7,Как погиб Пугачев?,"pushkin_14, pushkin_12, pushkin_11","pushkin_14, pushkin_12, pushkin_11",False
8,Кто был отцом Аркадия?,"turgenev_4, turgenev_14, turgenev_7","turgenev_new1, turgenev_4, turgenev_14",True
9,Как закончилась история Гринева?,"pushkin_14, pushkin_14, turgenev_24","pushkin_14, pushkin_14, turgenev_24",False


## 8. Mini-RAG: генерация ответа по найденным фрагментам

Соберём простой mini-RAG: по запросу ищем top-k фрагментов, формируем контекст и генерируем ответ (шаблонно). Примеры сохраняются в artifacts/rag_examples.csv.

In [32]:
# Mini-RAG: шаблонный генератор ответа

def mini_rag_answer(query, top_k=3):
    doc_ids = search_chunks_df(query, chunks_df_after, index_after, top_k=top_k)
    context_chunks = []
    for doc_id in doc_ids:
        chunk = chunks_df_after[chunks_df_after['doc_id'] == doc_id]['chunk_text'].values
        if len(chunk) > 0:
            context_chunks.append(chunk[0])
    context = '\n'.join(context_chunks)
    # Шаблонный ответ: просто пересказ найденного контекста
    answer = f"Ответ на ваш вопрос (по найденным фрагментам):\n{context[:400]}{'...' if len(context) > 400 else ''}"
    return answer, doc_ids

# Примеры работы mini-RAG
rag_examples = []
rag_questions = [
    "Кто такой Савельич?",
    "Что случилось с Базаровым?",
    "Главная тема романа Отцы и дети",
    "Как погиб Пугачев?",
    "Кто была Марья Ивановна?"
]
for q in rag_questions:
    answer, sources = mini_rag_answer(q, top_k=3)
    print(f"\nВопрос: {q}\nОтвет: {answer}\nИсточники: {sources}")
    rag_examples.append({'question': q, 'answer': answer, 'retrieved_sources': ', '.join(sources)})

# Сохраняем примеры
pd.DataFrame(rag_examples).to_csv('artifacts/rag_examples.csv', index=False, encoding='utf-8')


Вопрос: Кто такой Савельич?
Ответ: Ответ на ваш вопрос (по найденным фрагментам):
СЕРЖАНТ ГВАРДИИ. - Был бы гвардии он завтра ж капитан. - Того не надобно; пусть в армии послужит. - Изрядно сказано! пускай его потужит... ..................................... Да кто его отец? Княжнин.
ПРИСТУП. Голова моя, головушка, Голова послуживая! Послужила моя головушка Ровно тридцать лет и три года. Ах, не выслужила головушка Ни корысти себе, ни радости, Как ни слова себе доброго И
АРЕСТ. ...
Источники: ['pushkin_1', 'pushkin_7', 'pushkin_13']

Вопрос: Что случилось с Базаровым?
Ответ: Ответ на ваш вопрос (по найденным фрагментам):
Прошло шесть месяцев. Стояла белая зима с жестокою тишиной безоблачных морозов, плотным, скрипучим снегом, розовым инеем на деревьях, бледно-изумрудным небом, шапками дыма над трубами, клубами пара из мгновенно раскрытых дверей,
НЕЗВАНЫЙ ГОСТЬ. Незваный гость хуже татарина. Пословица. Площадь опустела. Я всё стоял на одном месте, и не мог привести в порядок мысли, смущ

В целом всё нормально ответил

## 9. Краткий анализ ошибок mini-RAG

Прокомментируем 2-4 неудачных или пограничных случая: где retrieval или генерация ответа сработали не идеально.

**Примеры ошибок и комментарии:**

- Для вопроса "Кто такая Марья Ивановна?" retrieval иногда возвращает не тот чанк, где она представлена, а фрагмент с упоминанием её имени без описания. Причина: слишком общий запрос и короткие чанки.
- Вопрос "Что случилось с Базаровым?" иногда приводит к выдаче фрагментов, где Базаров только упоминается, но не раскрывается его судьба. Причина: релевантный контекст размазан по нескольким главам.
- Для "Как погиб Пугачев?" retrieval может вернуть чанк с описанием событий до или после гибели, но не сам момент. Причина: формулировка вопроса и структура исходного текста.
- В целом, шаблонный генератор не всегда формирует связный ответ, если релевантный контекст разбит между чанками или retrieval ошибается.

**Вывод:**
- Ошибки чаще связаны с формулировкой запроса, размером чанка и ограничениями retrieval.
- Для повышения качества mini-RAG стоит использовать более сложные методы генерации и улучшать чанкинг.